# Tile Mosaic Image Acquisitions (Opera Phenix)

This notebook demonstrates the pipeline for compiling individual Opera Phenix acquisition fields into contiguous mosaic images. This is a critical prerequisite step for accurately tracking Mtb-infected macrophages that migrate across tile boundaries during long-term time-lapses.

Using an example dataset (`../data/untiled_images/`) representing a down-scaled multi-tile field of view, this notebook covers:

### Contents
1. **Load Acquisition Metadata:** Extracting acquisition parameters from the microscope's `Index.idx.xml`.
2. **Compile the Mosaic:** Compiling the multi-dimensional array using the `macrohet.tile` module.
3. **Compute and Visualise:** Computing the Dask array into memory for inspection via Napari.
4. **Save Compiled Mosaic Image Array:** Saving the stitched array to a standard bio-imaging format.
   * A) Save as OME-Zarr
     * i) Save In-Memory Array
     * ii) Stream Dask Array (Lazy Save)
   * B) Save as OME-TIFF

In [1]:
import napari
import numpy as np
import tifffile as tiff
import zarr

from tessell8er import dataio, tile

### 1. Load Acquisition Metadata
The `dataio` module parses the XML file generated by the Opera Phenix system to extract the spatial coordinates, timepoints, and channel metadata required for accurate physical stitching.

In [2]:
metadata = dataio.read_harmony_metadata('../../macro_organisation/macrohet/data/untiled_images/Index.idx.xml')
metadata.head()

Parsing Harmony Metadata:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

Metadata extraction complete.


,id,State,URL,Row,Col,FieldID,PlaneID,TimepointID,ChannelID,FlimID,...,PositionZ,AbsPositionZ,MeasurementTimeOffset,AbsTime,MainExcitationWavelength,MainEmissionWavelength,ObjectiveMagnification,ObjectiveNA,ExposureTime,OrientationMatrix
0,0305K1F1P1R1,Ok,r03c05f01p01-ch1sk1fk1fl1.tiff,3,5,1,1,0,1,1,...,0,0.135621503,0,2021-04-16T19:11:18.61+01:00,488,522,40,1.1,0.1,"[[0.990860,0,0,-15.9],[0,-0.990860,0,-44.8],[0..."
1,0305K1F1P1R2,Ok,r03c05f01p01-ch2sk1fk1fl1.tiff,3,5,1,1,0,2,1,...,0,0.135621503,0,2021-04-16T19:11:18.61+01:00,640,706,40,1.1,0.2,"[[0.990860,0,0,-15.9],[0,-0.990860,0,-44.8],[0..."
2,0305K1F1P2R1,Ok,r03c05f01p02-ch1sk1fk1fl1.tiff,3,5,1,2,0,1,1,...,2E-06,0.1356235,0,2021-04-16T19:11:18.89+01:00,488,522,40,1.1,0.1,"[[0.990860,0,0,-15.9],[0,-0.990860,0,-44.8],[0..."
3,0305K1F1P2R2,Ok,r03c05f01p02-ch2sk1fk1fl1.tiff,3,5,1,2,0,2,1,...,2E-06,0.1356235,0,2021-04-16T19:11:18.89+01:00,640,706,40,1.1,0.2,"[[0.990860,0,0,-15.9],[0,-0.990860,0,-44.8],[0..."
4,0305K1F1P3R1,Ok,r03c05f01p03-ch1sk1fk1fl1.tiff,3,5,1,3,0,1,1,...,4E-06,0.135625601,0,2021-04-16T19:11:19.17+01:00,488,522,40,1.1,0.1,"[[0.990860,0,0,-15.9],[0,-0.990860,0,-44.8],[0..."


### 2. Compile the Mosaic
We use the `tile.compile_mosaic` function to build the lazy Dask array. You must explicitly define the target well (via `row` and `col`) and the physical grid dimensions of the acquisition (`n_tile_cols` and `n_tile_rows`).

In [5]:
image_dir = '../../macro_organisation/macrohet/data/untiled_images'

images = tile.compile_mosaic(
    image_dir=image_dir, 
    metadata=metadata, 
    row=3, 
    col=5,  
    n_tile_cols=3, 
    n_tile_rows=3
)

images

AttributeError: 'Polygon' object has no attribute 'fuse_info'

### 3. Compute and Visualise
The `images` variable is currently a lazy Dask array. To view the stitched result in Napari, we call `.compute()` to process the stitching logic and load the final array into system RAM.

In [10]:
%%time
loaded_images = images.compute() 

CPU times: user 12min 24s, sys: 22.2 s, total: 12min 46s
Wall time: 1min 44s


In [19]:
viewer = napari.Viewer(title = 'Fully loaded mosaic compilation')
viewer.add_image(loaded_images, channel_axis=1, colormap=['green', 'magenta'])

[<Image layer 'Image' at 0x79b9d4d98220>,
 <Image layer 'Image [1]' at 0x79b9d5c096f0>]

### 4. Save Compiled Mosaic Image Array

We offer two formats for saving the compiled multi-dimensional image array (T, C, Z, Y, X): **OME-Zarr** and **OME-TIFF**. 

* **OME-Zarr (Recommended):** A modern, cloud-optimised format that supports chunking and lazy loading. We provide two methods for saving as Zarr depending on your hardware constraints.
* **OME-TIFF:** A highly compatible, traditional format for bio-image analysis.

First, we calculate the physical pixel scales from the Harmony metadata to embed into our chosen file format.

In [20]:
# Calculate pixel scales
res_x = float(metadata['ImageResolutionX'].astype(float).iloc[0])
res_y = float(metadata['ImageResolutionY'].astype(float).iloc[0])
z_vals = metadata['PositionZ'].astype(float).unique()
res_z = float(np.ptp(z_vals) / (len(z_vals) - 1)) if len(z_vals) > 1 else 1.0

# Define the minimal OME-NGFF metadata
ome_ngff_metadata = {
    "multiscales": [
        {
            "version": "0.4",
            "axes": [
                {"name": "t", "type": "time"},
                {"name": "c", "type": "channel"},
                {"name": "z", "type": "space"},
                {"name": "y", "type": "space"},
                {"name": "x", "type": "space"}
            ],
            "datasets": [
                {
                    "path": "0", 
                    "coordinateTransformations": [{"type": "scale", "scale": [1.0, 1.0, res_z, res_y, res_x]}]
                }
            ]
        }
    ]
}

#### A) Save as OME-Zarr

**i) Save In-Memory Array**
If you have already loaded the array into your system memory as `loaded_images`, run the following cell.

In [21]:
# Create the Zarr store and root group
store = zarr.DirectoryStore('../data/example_data.zarr')
root = zarr.group(store=store, overwrite=True)

# Attach the OME-NGFF metadata to the root
root.attrs.update(ome_ngff_metadata)

# Write the loaded_images array directly to the '0' dataset path
root.create_dataset(
    '0',
    data=loaded_images,
    chunks=(1, 1, 1, 256, 256),
    compressor=None
)

<zarr.core.Array '/0' (75, 2, 3, 1199, 1199) uint16>

**ii) Stream Dask Array (Lazy Save)**
If you are working with a dataset larger than your available RAM, you can bypass `loaded_images` entirely. This method streams the original lazy `images` Dask array directly to the disk, calculating and saving chunk-by-chunk.

In [22]:
# Create the Zarr store and root group
store = zarr.DirectoryStore('../data/example_data.zarr')
root = zarr.group(store=store, overwrite=True)

# Attach the OME-NGFF metadata to the root
root.attrs.update(ome_ngff_metadata)

# Stream the Dask array directly to the '0' dataset path
images.to_zarr(
    url='../data/example_data.zarr',
    component='0',
    overwrite=True,
    compressor=None
)

#### 4B. Save as OME-TIFF

If you prefer a standard TIFF format, you can save the `loaded_images` array as an OME-TIFF. This includes physical scaling metadata to ensure compatibility with standard viewers like Napari or Fiji.

In [23]:
# Save to OME-TIFF with full spatial metadata
tiff.imwrite(
    '../data/example_data.ome.tiff',
    loaded_images,
    photometric='minisblack',
    ome=True,
    resolution=(1.0 / res_x, 1.0 / res_y),
    resolutionunit='MICROMETER',
    metadata={
        'axes': 'TCZYX',
        'PhysicalSizeX': res_x,
        'PhysicalSizeY': res_y,
        'PhysicalSizeZ': res_z,
        'PhysicalSizeXUnit': 'µm',
        'PhysicalSizeYUnit': 'µm',
        'PhysicalSizeZUnit': 'µm'
    },
    bigtiff=True
)